# TakeMeter — Fine-Tuning DistilBERT for Football Discourse Classification

Fine-tunes `distilbert-base-uncased` to classify football/soccer community comments into three
discourse-structural categories, and compares it against a zero-shot Groq LLaMA-3.3-70B baseline
on the same held-out test set.

| Label | Meaning |
|-------|---------|
| `analysis` | Structured argument backed by stats, tactics, or historical comparison. |
| `hot_take` | Bold opinion with no systematic evidence; banter, hyperbole, GOAT/rivalry takes. |
| `reaction` | Immediate emotional response to a specific event, time-anchored to the moment. |

**Runtime:** set it to a **T4 GPU** (Runtime → Change runtime type → T4 GPU).

**Before running:** make sure the latest commit (with `data/`) is pushed to GitHub so the clone
below pulls the dataset.

## 0. Setup

In [ ]:
# DistilBERT fine-tuning + Groq baseline dependencies.
# torch is preinstalled on Colab; we add the HF + sklearn + groq stack.
!pip install -q -U transformers datasets accelerate scikit-learn groq python-dotenv

In [ ]:
import numpy as np
import pandas as pd
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> T4 GPU.")

## 1. Load the dataset

Clone the repo to pull `data/train.csv`, `data/val.csv`, `data/test.csv`. If you've changed the
data locally, push first. (Alternatively, comment the clone out and upload the CSVs manually.)

In [ ]:
import os

REPO = "https://github.com/fortdominz/ai201-project3-takemeter.git"
if not os.path.exists("ai201-project3-takemeter"):
    !git clone -q {REPO}
%cd ai201-project3-takemeter

train_df = pd.read_csv("data/train.csv")
val_df   = pd.read_csv("data/val.csv")
test_df  = pd.read_csv("data/test.csv")

print("train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df))
print("\nTrain label counts:\n", train_df["label"].value_counts())
train_df.head()

In [ ]:
# Label <-> id maps (alphabetical, stable)
LABELS = sorted(train_df["label"].unique().tolist())
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
print(label2id)

for df in (train_df, val_df, test_df):
    df["label_id"] = df["label"].map(label2id)

## 2. Tokenize

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf(df):
    ds = Dataset.from_pandas(df[["text", "label_id"]].rename(columns={"label_id": "labels"}),
                             preserve_index=False)
    return ds

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_ds = to_hf(train_df).map(tokenize, batched=True)
val_ds   = to_hf(val_df).map(tokenize, batched=True)
test_ds  = to_hf(test_df).map(tokenize, batched=True)

train_ds, val_ds, test_ds

## 3. Fine-tune — learning-rate experiment

Small dataset, so we keep it short (5 epochs, batch 16) and compare **two learning rates**
(`2e-5` vs `5e-5`), selecting the one with the better validation macro-F1. This is the
hyperparameter decision documented in `planning.md`.

In [ ]:
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

def train_one(lr, seed=42):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)
    args = TrainingArguments(
        output_dir=f"runs/lr_{lr}",
        learning_rate=lr,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        seed=seed,
        fp16=torch.cuda.is_available(),
        logging_steps=10,
        report_to="none",
        save_total_limit=1,
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        tokenizer=tokenizer, data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    val_metrics = trainer.evaluate()
    return trainer, val_metrics

In [ ]:
experiments = {}
for lr in [2e-5, 5e-5]:
    print(f"\n========== learning_rate = {lr} ==========")
    trainer, val_metrics = train_one(lr)
    experiments[lr] = {"trainer": trainer, "val": val_metrics}
    print(f"val: acc={val_metrics['eval_accuracy']:.3f}  f1={val_metrics['eval_f1']:.3f}")

best_lr = max(experiments, key=lambda k: experiments[k]["val"]["eval_f1"])
best_trainer = experiments[best_lr]["trainer"]
print(f"\nBest learning rate by val macro-F1: {best_lr}")

## 4. Evaluate the fine-tuned model on the held-out test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred_out = best_trainer.predict(test_ds)
y_true = pred_out.label_ids
y_pred = np.argmax(pred_out.predictions, axis=-1)

ft_acc = accuracy_score(y_true, y_pred)
ft_report = classification_report(y_true, y_pred, target_names=LABELS, output_dict=True, zero_division=0)
print(f"Fine-tuned DistilBERT — test accuracy: {ft_acc:.3f}\n")
print(classification_report(y_true, y_pred, target_names=LABELS, zero_division=0))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

os.makedirs("outputs", exist_ok=True)
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS)
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Fine-tuned DistilBERT (test acc {ft_acc:.2f})")
plt.tight_layout()
plt.savefig("outputs/confusion_matrix.png", dpi=150)
plt.show()

### Misclassified examples
The 3 required error cases for the write-up.

In [ ]:
mis = test_df.reset_index(drop=True).copy()
mis["pred"] = [id2label[p] for p in y_pred]
mis_wrong = mis[mis["label"] != mis["pred"]][["text", "label", "pred"]]
print(f"{len(mis_wrong)} misclassified out of {len(mis)}\n")
for _, row in mis_wrong.head(5).iterrows():
    print(f"TRUE={row['label']:<9} PRED={row['pred']:<9} | {row['text'][:120]}")
mis_wrong.head(10)

## 5. Zero-shot baseline — Groq LLaMA-3.3-70B

Same test set, no fine-tuning — just the label definitions in the prompt (see `baseline_groq.py`).

Add your key via the Colab Secrets panel (🔑 in the left sidebar) as `GROQ_API_KEY`, then run.
Falls back to a prompt if the secret isn't set.

In [ ]:
# Load the Groq API key from Colab Secrets (preferred) or prompt for it.
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    from getpass import getpass
    GROQ_API_KEY = getpass("Enter your GROQ_API_KEY: ")

assert GROQ_API_KEY, "GROQ_API_KEY is required for the baseline."
print("Groq key loaded.")

In [ ]:
# Reuse the canonical baseline implementation from the repo.
from baseline_groq import run_baseline

baseline = run_baseline("data/test.csv", api_key=GROQ_API_KEY, pause=1.0)
print(f"\nBaseline accuracy: {baseline['accuracy']:.3f}")

## 6. Results & comparison

In [ ]:
import json

results = {
    "labels": LABELS,
    "test_size": len(test_df),
    "fine_tuned_distilbert": {
        "base_model": MODEL_NAME,
        "best_learning_rate": best_lr,
        "val_metrics_by_lr": {
            str(lr): {k: float(v) for k, v in exp["val"].items() if k.startswith("eval_")}
            for lr, exp in experiments.items()
        },
        "test_accuracy": float(ft_acc),
        "test_classification_report": ft_report,
        "confusion_matrix": {"labels": LABELS, "matrix": cm.tolist()},
    },
    "groq_zero_shot_baseline": {
        "model": baseline["model"],
        "test_accuracy": float(baseline["accuracy"]),
        "test_classification_report": baseline["classification_report"],
        "confusion_matrix": baseline["confusion_matrix"],
    },
}

with open("outputs/evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("Saved outputs/evaluation_results.json\n")
print(f"{'Model':<32}{'Test accuracy':>14}")
print("-" * 46)
print(f"{'Fine-tuned DistilBERT':<32}{ft_acc:>14.3f}")
print(f"{'Zero-shot LLaMA 3.3 70B':<32}{baseline['accuracy']:>14.3f}")

In [ ]:
# Download the artifacts (or commit them from Colab).
from google.colab import files
files.download("outputs/evaluation_results.json")
files.download("outputs/confusion_matrix.png")

## Next steps

- Drop `test_accuracy` for both models into the README results table.
- Save the two misclassified-example screenshots / rows for the write-up.
- (Stretch) Wrap the fine-tuned model in a Gradio interface.